In [1]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import os
import gc
import re
import random
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

print("Libraries imported successfully.")
print("PyTorch:", torch.__version__)
print("Transformers:", __import__("transformers").__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Total VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )


Libraries imported successfully.
PyTorch: 2.10.0+cu128
Transformers: 5.0.0
CUDA available: True
GPU: Tesla T4
Total VRAM: 14.56 GB


In [2]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

DATA_DIR = (
    "/kaggle/input/datasets/"
    "taufiqzahantushar/checksent-bn/"
    "CheckSent-BN"
)

TRAIN_PATH = os.path.join(DATA_DIR, "train_data.tsv")
TEST_PATH = os.path.join(DATA_DIR, "test_data.tsv")

MAX_NEW_TOKENS = 10

LABELS = [
    "NEGATIVE",
    "NEUTRAL",
    "POSITIVE"
]


In [3]:

# ============================================================
# 3. LOAD DATASET
# ============================================================

train_df = pd.read_csv(
    TRAIN_PATH,
    sep="\t"
)

test_df = pd.read_csv(
    TEST_PATH,
    sep="\t"
)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain sentiment distribution:")
print(train_df["Sentiment"].value_counts())

print("\nTest sentiment distribution:")
print(test_df["Sentiment"].value_counts())


Train shape: (9254, 6)
Test shape: (2314, 6)

Train sentiment distribution:
Sentiment
NEGATIVE    4573
NEUTRAL     3187
POSITIVE    1494
Name: count, dtype: int64

Test sentiment distribution:
Sentiment
NEGATIVE    1123
NEUTRAL      825
POSITIVE     366
Name: count, dtype: int64


In [4]:
# ============================================================
# 4. VALIDATE DATA
# ============================================================

required_columns = [
    "ID",
    "Date",
    "News Portal",
    "Headline",
    "Claim",
    "Sentiment"
]

assert all(
    column in train_df.columns
    for column in required_columns
)

assert all(
    column in test_df.columns
    for column in required_columns
)

assert train_df["Headline"].isna().sum() == 0
assert test_df["Headline"].isna().sum() == 0

assert train_df["Sentiment"].isna().sum() == 0
assert test_df["Sentiment"].isna().sum() == 0

assert set(test_df["Sentiment"].unique()) == set(LABELS)

print("\nDataset validation passed.")


Dataset validation passed.


In [5]:

# ============================================================
# 5. HUGGING FACE TOKEN
# ============================================================

from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

HF_TOKEN = user_secrets.get_secret("gptoss")

print("Hugging Face token loaded.")


Hugging Face token loaded.


In [6]:
# ============================================================
# 6. LOAD TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN
)

print("Tokenizer loaded successfully.")


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded successfully.


In [7]:
# ============================================================
# 7. LOAD MODEL
# ============================================================

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    device_map="auto",
    torch_dtype="auto"
)

model.eval()

print("Model loaded successfully.")
print("Model device:", model.device)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully.
Model device: cuda:0


In [8]:
# ============================================================
# 8. SELECT FEW-SHOT EXAMPLES
# ============================================================

few_shot_examples = {}

for label in LABELS:

    example = (
        train_df[
            train_df["Sentiment"] == label
        ]
        .iloc[0]
    )

    few_shot_examples[label] = {
        "headline": example["Headline"],
        "label": label
    }


print("\nFew-shot examples selected:\n")

for label in LABELS:

    print(
        f"{label}:"
    )

    print(
        few_shot_examples[label]["headline"]
    )

    print()


Few-shot examples selected:

NEGATIVE:
বিএসএফের মদতে অনুপ্রবেশ: মমতা, বাংলাকে অশান্ত করতে কেন্দ্রের ছক

NEUTRAL:
দৈর্ঘ্য মাত্র ৪ ইঞ্চি, উপত্যকায় জঙ্গি দমনে সেনার মারণাস্ত্র ‘কালো ভ্রমর’

POSITIVE:
বেঙ্গালুরুকে হারিয়ে অঘটন মহামেডানের, ডার্বির আগে বাড়তি অক্সিজেন পেল মোহনবাগান



In [9]:
# ============================================================
# 9. SENTIMENT PARSER
# ============================================================

def parse_sentiment(output):

    if output is None:
        return None

    output = output.strip().upper()

    output = output.replace("`", "")
    output = output.replace("*", "")

    if output == "NEGATIVE":
        return "NEGATIVE"

    if output == "NEUTRAL":
        return "NEUTRAL"

    if output == "POSITIVE":
        return "POSITIVE"

    matches = []

    for label in LABELS:

        if re.search(
            rf"\b{label}\b",
            output
        ):
            matches.append(label)

    if len(matches) == 1:
        return matches[0]

    return None

In [10]:
# ============================================================
# 10. FEW-SHOT PROMPT
# ============================================================

def build_few_shot_prompt(headline):

    return f"""
You are performing Bengali sentiment classification.

Classify the sentiment of the Bengali news headline into exactly
ONE of these labels:

NEGATIVE = The headline expresses negative sentiment,
negative events, criticism, conflict, loss, danger, failure,
or other negative emotional polarity.

NEUTRAL = The headline is factual, informational, descriptive,
or does not clearly express positive or negative sentiment.

POSITIVE = The headline expresses positive sentiment,
success, achievement, benefit, celebration, praise,
or other positive emotional polarity.

Here are three labeled examples:

Example 1:
Headline:
{few_shot_examples["NEGATIVE"]["headline"]}

Label:
NEGATIVE


Example 2:
Headline:
{few_shot_examples["NEUTRAL"]["headline"]}

Label:
NEUTRAL


Example 3:
Headline:
{few_shot_examples["POSITIVE"]["headline"]}

Label:
POSITIVE


Now classify the following headline.

Return ONLY ONE label.

Allowed labels:
NEGATIVE
NEUTRAL
POSITIVE

Headline:
{headline}

Label:
""".strip()

In [11]:

# ============================================================
# 11. SINGLE SAMPLE PREDICTION
# ============================================================

def predict_sentiment(headline):

    messages = [
        {
            "role": "user",
            "content": build_few_shot_prompt(
                headline
            )
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    inputs = inputs.to(model.device)

    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None
        )

    generated_tokens = outputs[0, input_length:]

    generated_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    prediction = parse_sentiment(
        generated_text
    )

    return prediction, generated_text

In [12]:
# ============================================================
# 12. TEST SAMPLE PREDICTIONS
# ============================================================

sample_indices = [
    0,
    1,
    2,
    10,
    20
]

print("Sample predictions:\n")

for idx in sample_indices:

    headline = test_df.iloc[idx]["Headline"]
    actual = test_df.iloc[idx]["Sentiment"]

    prediction, raw_output = predict_sentiment(
        headline
    )

    print("=" * 80)
    print("Index:", idx)
    print("Headline:", headline)
    print("Actual:", actual)
    print("Prediction:", prediction)
    print("Raw output:", repr(raw_output))

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Sample predictions:

Index: 0
Headline: বেঙ্গালুরু-দমদম বিমানে বোমাতঙ্ক! সোশাল মিডিয়ায় লাগাতার হুমকি পোস্টে আতঙ্ক
Actual: NEGATIVE
Prediction: NEGATIVE
Raw output: 'NEGATIVE'
Index: 1
Headline: নেপালে নদীতে পড়ল ভারতীয় পর্যটকবোঝাই বাস, মৃত অন্তত ১৪
Actual: NEGATIVE
Prediction: NEGATIVE
Raw output: 'NEGATIVE'
Index: 2
Headline: পুজো হবে নির্বিঘ্নে, আশ্বাস ইউনুস সরকারের, মহালয়ার প্রস্তুতি বাংলাদেশে
Actual: NEUTRAL
Prediction: NEUTRAL
Raw output: 'NEUTRAL'
Index: 10
Headline: প্রবল বর্ষণে বিপর্যস্ত মেঘালয়, হড়পা বান ও ধসে মৃত ১০ জন
Actual: NEGATIVE
Prediction: NEGATIVE
Raw output: 'NEGATIVE'
Index: 20
Headline: ইউটিউবে রোনাল্ডোর সাতকাহন, কোন সাত কারণে সিআর সেভেনের চ্যানেলে বুঁদ ভক্তরা?
Actual: POSITIVE
Prediction: NEUTRAL
Raw output: 'NEUTRAL'


In [13]:
# ============================================================
# 13. FULL FEW-SHOT EVALUATION
# ============================================================

few_shot_predictions = []
few_shot_raw_outputs = []

print("\nStarting Few-shot evaluation...\n")

for idx, headline in enumerate(
    tqdm(
        test_df["Headline"],
        total=len(test_df),
        desc="Few-shot"
    )
):

    try:

        prediction, raw_output = predict_sentiment(
            headline
        )

    except Exception as e:

        print(
            f"\nError at index {idx}: {e}"
        )

        prediction = None
        raw_output = ""

    few_shot_predictions.append(
        prediction
    )

    few_shot_raw_outputs.append(
        raw_output
    )

print("\nFew-shot inference completed.")

print(
    "Total predictions:",
    len(few_shot_predictions)
)

print(
    "Unknown predictions:",
    sum(
        p is None
        for p in few_shot_predictions
    )
)


Starting Few-shot evaluation...



Few-shot:   0%|          | 0/2314 [00:00<?, ?it/s]


Few-shot inference completed.
Total predictions: 2314
Unknown predictions: 0


In [14]:
# ============================================================
# 14. PREPARE PREDICTIONS
# ============================================================

y_true = test_df["Sentiment"].tolist()

y_pred = [
    prediction if prediction in LABELS
    else "UNKNOWN"
    for prediction in few_shot_predictions
]

print("\nPrediction distribution:")
print(
    pd.Series(y_pred).value_counts()
)



Prediction distribution:
NEGATIVE    1356
NEUTRAL      529
POSITIVE     429
Name: count, dtype: int64


In [15]:
# ============================================================
# 15. VALID PREDICTIONS
# ============================================================

valid_indices = [
    i
    for i, prediction in enumerate(y_pred)
    if prediction in LABELS
]

valid_y_true = [
    y_true[i]
    for i in valid_indices
]

valid_y_pred = [
    y_pred[i]
    for i in valid_indices
]

In [16]:
# ============================================================
# 16. METRICS
# ============================================================

accuracy = accuracy_score(
    valid_y_true,
    valid_y_pred
)

precision, recall, macro_f1, _ = (
    precision_recall_fscore_support(
        valid_y_true,
        valid_y_pred,
        labels=LABELS,
        average="macro",
        zero_division=0
    )
)

In [17]:
# ============================================================
# 17. MAIN RESULTS
# ============================================================

print("\n" + "=" * 60)
print("QWEN2.5-3B-INSTRUCT — FEW-SHOT RESULTS")
print("=" * 60)

print(
    f"Accuracy : {accuracy:.4f} "
    f"({accuracy * 100:.2f}%)"
)

print(
    f"Precision: {precision:.4f} "
    f"({precision * 100:.2f}%)"
)

print(
    f"Recall   : {recall:.4f} "
    f"({recall * 100:.2f}%)"
)

print(
    f"Macro F1 : {macro_f1:.4f} "
    f"({macro_f1 * 100:.2f}%)"
)

print(
    f"Unknown  : {len(y_true) - len(valid_y_true)}"
)


QWEN2.5-3B-INSTRUCT — FEW-SHOT RESULTS
Accuracy : 0.6936 (69.36%)
Precision: 0.6619 (66.19%)
Recall   : 0.6567 (65.67%)
Macro F1 : 0.6463 (64.63%)
Unknown  : 0


In [18]:
# ============================================================
# 18. CLASS-WISE RESULTS
# ============================================================

print("\n" + "=" * 60)
print("CLASS-WISE RESULTS")
print("=" * 60)

print(
    classification_report(
        valid_y_true,
        valid_y_pred,
        labels=LABELS,
        target_names=LABELS,
        digits=4,
        zero_division=0
    )
)


CLASS-WISE RESULTS
              precision    recall  f1-score   support

    NEGATIVE     0.7345    0.8869    0.8035      1123
     NEUTRAL     0.7221    0.4630    0.5643       825
    POSITIVE     0.5291    0.6202    0.5711       366

    accuracy                         0.6936      2314
   macro avg     0.6619    0.6567    0.6463      2314
weighted avg     0.6976    0.6936    0.6815      2314



In [19]:
# ============================================================
# 19. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    valid_y_true,
    valid_y_pred,
    labels=LABELS
)

cm_df = pd.DataFrame(
    cm,
    index=LABELS,
    columns=LABELS
)

print("\nConfusion Matrix:")
display(cm_df)



Confusion Matrix:


,NEGATIVE,NEUTRAL,POSITIVE
NEGATIVE,996,79,48
NEUTRAL,289,382,154
POSITIVE,71,68,227


In [20]:
# ============================================================
# 20. FINAL RESULT TABLE
# ============================================================

few_shot_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "Macro F1"
    ],
    "Score": [
        accuracy,
        precision,
        recall,
        macro_f1
    ]
})

print("\nFinal Few-shot Results:")
display(few_shot_results)


Final Few-shot Results:


,Metric,Score
0,Accuracy,0.693604
1,Precision,0.661923
2,Recall,0.656720
3,Macro F1,0.646291
